# **01 LangChain 기초**

### 학습 내용
1. Chat Models 초기화
2. Messages 이해
3. 대화 히스토리 관리
4. 스트리밍 응답

## 0. 환경 설정

- OpenAI API Key 발급: https://platform.openai.com/api-keys

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.environ.get("OPENAI_API_KEY"):
    print("✓ API Key가 설정되었습니다.")

✓ API Key가 설정되었습니다.


## 1. Chat Models(LLM) 초기화

### `init_chat_model()`

참고: https://docs.langchain.com/oss/python/langchain/models

In [3]:
# 방법 1: init_chat_model() 사용
from langchain.chat_models import init_chat_model

# 모델 초기화
llm = init_chat_model(
    "gpt-5.4-mini",  # 또는 "gpt-4o", "claude-sonnet-4-5-20250929"
)

# 모델 호출
response = llm.invoke("LangChain이란 무엇인지 3문장으로 설명해줘.")
print(response.content)

LangChain은 대규모 언어 모델(LLM)을 활용한 애플리케이션을 쉽게 만들 수 있도록 도와주는 개발 프레임워크입니다.  
프롬프트 관리, 외부 데이터 연동, 여러 단계의 작업 흐름 구성 등을 간편하게 지원합니다.  
즉, 챗봇, 검색 보조, 자동화 에이전트 같은 LLM 기반 서비스를 빠르게 개발할 수 있게 해주는 도구입니다.


### 주요 파라미터

| 파라미터 | 설명 | 기본값 |
|---------|------|--------|
| `model` | 사용할 모델 이름 (예: "gpt-4o-mini") | 필수 |
| `temperature` | 창의성 조절 (0.0=결정적, 1.0=창의적) | 모델별 상이 |
| `max_tokens` | 생성할 최대 토큰 수 | 모델별 상이 |
| `timeout` | API 요청 타임아웃 (초) | 없음 |
| `max_retries` | 실패 시 재시도 횟수 (지수 백오프 적용) | 6 |

## 2. 메시지 (Messages) 이해

**메시지(Message)** 는 LangChain에서 대화를 관리하는 핵심 개념입니다.

### 2-1. 메시지 구조

1. **Role (역할)** - 메시지를 보낸 주체 (`system`, `user`, `assistant`)
2. **Content (내용)** - 메시지 텍스트
3. **Metadata (메타데이터)** - 추가 정보 (ID, 타임스탬프, 토큰 사용량 등)

In [4]:
conversation = [
    {"role": "system", "content": "당신은 한국어를 프랑스어로 번역하는 번역가입니다."},
    {"role": "user", "content": "번역: 나는 프로그래밍을 좋아합니다."},
    {"role": "assistant", "content": "J'adore la programmation."},
    {"role": "user", "content": "번역: 나는 애플리케이션을 만드는 것을 좋아합니다."}
]

response = llm.invoke(conversation)
print(response)  # AIMessage("J'adore créer des applications.")

content="J'aime créer des applications." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 70, 'total_tokens': 79, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EA8YdjJ57U8A8JRgYubQBqigGfjPe', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019fdaf4-794c-7542-b798-eefb0a6704f7-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 70, 'output_tokens': 9, 'total_tokens': 79, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [5]:
response.content

"J'aime créer des applications."

### 2-2. 문자열 vs 메시지 객체

| 항목 | 문자열 | 메시지 객체 |
|-----|--------------|---------------|
| 사용법 | 간단함 | Message 클래스 사용 |
| 역할 지정 | 기본 1개만 | 여러 역할, 메타데이터 가능 |
| 적합한 경우 | 단순 질문 | 복잡한 대화 |

참고: https://docs.langchain.com/oss/python/langchain/messages

**문자열 사용 예시:**
```python
response = model.invoke("Write a haiku about spring")
```

**메시지 객체 사용 예시:**
```python
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage("You are a poetry expert"),
    HumanMessage("Write a haiku about spring"),
    AIMessage("Cherry blossoms bloom...")
]
response = model.invoke(messages)
```

### 2-3. 메시지 타입 (Message Types)


**1) SystemMessage - 시스템 메시지**
- 모델의 행동을 정의합니다. 모델에게 역할을 부여합니다.

**2) HumanMessage - 사용자 메시지**
- 사용자의 입력입니다. 질문, 요청, 명령 등을 나타냅니다.

**3) AIMessage - AI 응답**
- 모델의 응답입니다. 추가로 tool_calls, usage_metadata 등을 포함할 수 있습니다.

**4) ToolMessage - 도구 결과**
- Tool 실행 결과를 나타냅니다.

In [6]:
# 문자열 사용 (간단한 방법)
response = llm.invoke("AI가 무엇인가요?")
print("=== 문자열 사용 ===")
print(response.content)
print(f"응답 타입: {type(response)}")

=== 문자열 사용 ===
AI는 **인공지능(Artificial Intelligence)**의 줄임말로, **사람처럼 학습하고 판단하거나 문제를 해결하도록 만든 컴퓨터 기술**을 말합니다.

예를 들면:
- **음성 인식**: “시리야, 음악 틀어줘”
- **추천 시스템**: 유튜브/넷플릭스가 취향에 맞는 콘텐츠 추천
- **챗봇**: 사람처럼 대화하는 프로그램
- **이미지 인식**: 사진 속 고양이, 사람, 사물을 구분

쉽게 말해, **컴퓨터가 사람의 지능 일부를 흉내 내는 기술**이라고 볼 수 있습니다.

원하시면 제가  
1) **아주 쉽게 설명**하거나  
2) **AI의 종류와 예시**를 더 자세히 알려드릴게요.
응답 타입: <class 'langchain_core.messages.ai.AIMessage'>


In [7]:
# 메시지 객체 사용 (자세한 방법)
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage("당신은 친절한 AI 어시스턴트입니다."),
    HumanMessage("AI가 무엇인가요?")
]
response = llm.invoke(messages)
print("=== 메시지 객체 사용 ===")
print(response.content)
print(f"응답 타입: {type(response)}")

=== 메시지 객체 사용 ===
AI는 **인공지능(Artificial Intelligence)**의 줄임말로,  
사람처럼 **학습하고 판단하고 문제를 해결하는 일을 컴퓨터가 하도록 만드는 기술**을 말합니다.

예를 들면:
- 음성 비서가 말을 알아듣는 것
- 사진 속 얼굴을 인식하는 것
- 추천 영상이나 상품을 골라주는 것
- 글을 생성하거나 번역하는 것

쉽게 말해, **사람의 지능이 필요한 작업을 기계가 일부 대신하게 하는 기술**이라고 볼 수 있습니다.

원하시면 제가 **AI의 종류**, **작동 원리**, 또는 **일상 속 AI 예시**도 쉽게 설명해드릴게요.
응답 타입: <class 'langchain_core.messages.ai.AIMessage'>


In [8]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# SystemMessage로 역할 정의 - 종료 시까지 유지됨
system_msg = SystemMessage("""
당신은 10년 경력의 Python 개발자입니다.
사용자의 질문에 정확하고 친절하게 답변하세요.
코드 예시를 포함하세요.
""")

messages = [
    system_msg,
    HumanMessage(content="Python에서 리스트 컴프리헨션을 설명해주세요.")
]

response = llm.invoke(messages)
print(response.content)
print()

리스트 컴프리헨션(List Comprehension)은 **리스트를 간결하게 만드는 문법**입니다.  
반복문과 조건문을 한 줄에 압축해서 새로운 리스트를 만들 수 있어요.

---

## 1) 기본 형태

```python
[표현식 for 변수 in 반복가능한객체]
```

예를 들면, 0부터 4까지 숫자의 제곱 리스트를 만들 수 있습니다.

```python
squares = [x * x for x in range(5)]
print(squares)
# [0, 1, 4, 9, 16]
```

위 코드는 아래와 같은 `for`문과 같은 의미입니다.

```python
squares = []
for x in range(5):
    squares.append(x * x)
```

---

## 2) 조건문 포함하기

조건을 넣어서 특정 값만 골라낼 수 있습니다.

```python
evens = [x for x in range(10) if x % 2 == 0]
print(evens)
# [0, 2, 4, 6, 8]
```

이것도 일반 `for`문으로 쓰면 다음과 같습니다.

```python
evens = []
for x in range(10):
    if x % 2 == 0:
        evens.append(x)
```

---

## 3) 조건에 따라 값 바꾸기

리스트 컴프리헨션 안에서 `if ... else ...`를 사용해 값을 바꿀 수도 있습니다.

```python
result = ["짝수" if x % 2 == 0 else "홀수" for x in range(5)]
print(result)
# ['짝수', '홀수', '짝수', '홀수', '짝수']
```

주의할 점은 이 경우 문법이:

```python
[참일때값 if 조건 else 거짓일때값 for 변수 in 반복가능한객체]
```

형태라는 것입니다.

---

## 4) 중첩 for문도 가능

```python
pairs = [(x, y) for x in range(2) for y

### 📖 과제 1: 시스템 프롬프트로 챗봇의 역할 정의하기

SystemMessage로 시스템 프롬프트(페르소나, 역할, 규칙 등 정의)를 작성하고 답변이 어떻게 달라지는 지 확인해봅시다.

다양한 도메인의 전문가로 AI의 역할을 다시 정의하거나, 더 나은 답변을 출력하기 위해 시스템 프롬프트를 바꿔보세요.

- 주제 (선생님 / 의사 / 소설가 / 요리사 / 법률 상담가 등)
- 말투 (친근하게 / 엄격하게 / 비유 많이 사용)
- 답변 형식 (항상 단계별 / 항상 예시 포함 / 항상 해시태그 포함)
- 대상 독자 (초등학생 / 비전공자 / 전문가)


In [9]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# TODO: SystemMessage 작성
system_msg = SystemMessage("""
이 곳에 시스템 프롬프트를 작성해보세요.
""")

messages = [
    system_msg,
    HumanMessage(content="이 곳에 사용자의 질문을 작성해보세요.")
]

response = llm.invoke(messages)
print(response.content)
print()

안녕하세요! 무엇을 도와드릴까요?



## 3. 대화 히스토리 관리

Chat Model은 이전 대화를 기억하지 않습니다. 필요한 대화 내용은 개발자가 직접 전달해야 합니다.

### 3-1. 대화 관리 흐름 이해하기

1. 사용자 메시지를 HumanMessage로 변환
2. 모델에 전달
3. 응답을 AIMessage로 저장
4. 다음 요청 시 이전 대화를 함께 전달

In [10]:
# 대화 히스토리 축적
chat_history = []

# 첫 번째 대화
print("=== 대화 1 ===")
user_msg_1 = HumanMessage(content="내 이름은 찰리야.")
chat_history.append(user_msg_1)

response_1 = llm.invoke(chat_history)
print(f"User: {user_msg_1.content}")
print(f"AI: {response_1.content}")

# AI 응답 저장
chat_history.append(response_1)
print(f"\n대화 히스토리: {len(chat_history)}개 메시지")
for i, msg in enumerate(chat_history, 1):
    print(f"{i}. [{msg.type}] {msg.content}")

=== 대화 1 ===
User: 내 이름은 찰리야.
AI: 반가워, 찰리!  
무엇을 도와줄까?

대화 히스토리: 2개 메시지
1. [human] 내 이름은 찰리야.
2. [ai] 반가워, 찰리!  
무엇을 도와줄까?


In [11]:
# 두 번째 대화
print("\n=== 대화 2 ===")
user_msg_2 = HumanMessage(content="내 이름이 뭐였어?")
chat_history.append(user_msg_2)

response_2 = llm.invoke(chat_history)
print(f"User: {user_msg_2.content}")
print(f"AI: {response_2.content}")
chat_history.append(response_2)

# 히스토리 확인
print(f"\n대화 히스토리: {len(chat_history)}개 메시지")
for i, msg in enumerate(chat_history, 1):
    print(f"{i}. [{msg.type}] {msg.content}")


=== 대화 2 ===
User: 내 이름이 뭐였어?
AI: 네 이름은 찰리야.

대화 히스토리: 4개 메시지
1. [human] 내 이름은 찰리야.
2. [ai] 반가워, 찰리!  
무엇을 도와줄까?
3. [human] 내 이름이 뭐였어?
4. [ai] 네 이름은 찰리야.


### 3-2. 멀티턴 동작

> **멀티턴(Multi-turn)** 은 AI나 챗봇이 이전 대화의 맥락과 이력을 기억한 채 여러 번 질문과 답변을 주고받는 상호작용 방식을 말합니다.

다음은 멀티턴을 기반으로 동작하는 간단한 챗봇입니다. 직접 대화를 나눠보며, 챗봇이 대화를 잘 기억하는 지 확인해봅시다.

In [12]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from IPython.display import Markdown, display

conversation = [
    SystemMessage(content="당신은 여행 가이드입니다. 여행 장소에 대한 여행 계획이나 참고할만한 정보를 제공하세요."),
]

print("\n🧭 여행 가이드 챗봇 시작\n")
print("-" * 50)

while True:
    user_input = input("🧑 User: ")
    print(f"🧑 User: : {user_input}")
    if user_input.lower() in ["q", "exit", "quit"]:
        print("\n👋 대화를 종료합니다.")
        break

    # user 메시지 저장
    conversation.append(HumanMessage(content=user_input))

    response = llm.invoke(conversation)

    # AI 메시지 저장
    conversation.append(AIMessage(content=response.content))

    # 출력 (Markdown 렌더링)
    print("\n🤖 AI:")
    display(Markdown(response.content))
    print("-" * 50)


🧭 여행 가이드 챗봇 시작

--------------------------------------------------
🧑 User: : 안녕

🤖 AI:


안녕하세요! 여행 가이드입니다.  
어느 여행지에 대한 계획이나 정보가 필요하신가요?

예를 들면:
- 도시/국가 여행 일정
- 맛집/카페 추천
- 교통편
- 숙소 지역 추천
- 관광지 코스
- 예산별 여행 팁

원하시는 여행지를 말씀해 주시면 바로 도와드릴게요.

--------------------------------------------------
🧑 User: : 도쿄 3박 4일 여행 코스 추천해줘

🤖 AI:


좋아요! 도쿄 3박 4일은 **도심 핵심 + 근교 감성 + 쇼핑/맛집**을 섞으면 만족도가 높아요.  
처음 가는 분 기준으로, 이동 동선이 무리 없도록 추천해드릴게요.

---

## 도쿄 3박 4일 추천 코스

### 1일차: 도쿄 도착 + 시부야/하라주쿠/신주쿠
**핵심 분위기:** 도쿄 첫인상, 젊은 거리, 쇼핑, 야경

- **오전/낮**: 공항 도착 → 숙소 체크인
- **오후**: 시부야
  - 시부야 스크램블 교차로
  - 시부야 스카이(가능하면 사전 예약 추천)
  - 쇼핑: 스크램블 스퀘어, 파르코, 히카리에
- **저녁**: 하라주쿠/오모테산도 산책
  - 다케시타 거리
  - 오모테산도 거리 구경
- **밤**: 신주쿠
  - 도쿄 도청사 전망대(무료)
  - 가부키초, 골든가이 분위기 보기

**추천 포인트**
- 첫날은 너무 빡빡하게 넣지 말고 도심 분위기 적응용으로 구성
- 야경은 시부야 스카이 또는 도쿄 도청사 중 하나만 가도 충분해요

---

### 2일차: 아사쿠사/우에노/아키하바라
**핵심 분위기:** 전통 + 시장 + 오타쿠/전자상가

- **오전**: 아사쿠사
  - 센소지
  - 나카미세 거리
  - 카미나리몬
- **점심**: 아사쿠사 근처에서 텐동/소바/장어덮밥
- **오후**: 우에노
  - 우에노 공원
  - 국립박물관 또는 동물원(취향 따라 선택)
  - 아메요코 시장 구경
- **저녁**: 아키하바라
  - 전자상가, 애니메이션 숍, 게임센터
  - 메이드카페나 이색 카페 체험 가능

**추천 포인트**
- 전통적인 도쿄와 현대적인 도쿄를 하루에 함께 보기 좋아요
- 사진 찍기 좋은 곳이 많아서 만족도 높습니다

---

### 3일차: 하코네 또는 디즈니랜드 중 선택
이 날은 취향에 따라 크게 갈려요.

#### A안: 자연·온천·풍경 좋아하면 **하코네**
- 도쿄 근교로 당일치기 가능
- 오와쿠다니, 아시노코 호수, 로프웨이, 온천 료칸 체험
- 조금 여유 있고 일본 감성 강한 여행에 좋아요

#### B안: 테마파크 좋아하면 **도쿄 디즈니랜드/디즈니씨**
- 하루 종일 즐기기 좋음
- 커플/가족/친구 여행에 특히 추천
- 디즈니씨는 좀 더 어른 취향, 디즈니랜드는 클래식한 분위기

#### C안: 도쿄 시내 여유 코스
- 긴자 쇼핑
- 츠키지 장외시장
- 도쿄역 주변
- 황거 외곽 산책
- 이 날을 쇼핑/카페/미술관으로 써도 좋아요

---

### 4일차: 츠키지/긴자/도쿄역 + 귀국
**핵심 분위기:** 마지막 날 가볍게, 선물 쇼핑, 식사

- **아침**: 츠키지 장외시장
  - 초밥, 계란말이, 해산물 덮밥
- **오전**: 긴자
  - 백화점, 명품 거리, 기념품 쇼핑
  - 도쿄역 야에스/도쿄 캐릭터 스트리트
- **점심 후**: 공항 이동

**추천 포인트**
- 귀국일은 공항 이동 시간을 넉넉히 잡는 게 중요해요
- 나리타/하네다에 따라 이동 시간이 크게 달라집니다

---

## 숙소 추천 지역
처음 가는 분이면 아래 지역이 편해요.

### 1) **신주쿠**
- 장점: 교통 편리, 먹을 곳 많음, 공항 접근성 좋음
- 단점: 사람이 많고 복잡함

### 2) **시부야**
- 장점: 젊고 세련된 분위기, 쇼핑/카페 좋음
- 단점: 숙소 가격이 다소 높을 수 있음

### 3) **우에노**
- 장점: 비교적 가성비 좋고 공항 이동 편리
- 단점: 밤 분위기는 신주쿠/시부야보다 덜 화려함

---

## 여행 팁
- **교통패스**: 도쿄는 무리하게 패스보다 **스이카/파스모 교통카드**가 편해요
- **시부야 스카이, 디즈니, 인기 맛집**은 사전 예약 권장
- **현금 조금은 필요**하지만 카드도 대부분 잘 됩니다
- **걷는 양이 많으니 편한 신발** 필수
- **구글맵**이 가장 유용해요

---

원하시면 제가 다음 중 하나로 더 구체화해드릴게요:
1. **예산별 코스**  
2. **커플 여행용 코스**  
3. **가족 여행용 코스**  
4. **맛집 중심 코스**  
5. **쇼핑 중심 코스**  

원하시는 스타일 말씀해주시면 그에 맞춰 **시간대별 일정표**로 짜드릴게요.

--------------------------------------------------
🧑 User: : 2박3일로 줄일래

🤖 AI:


물론이죠!  
도쿄 **2박 3일**이면 핵심만 잘 묶어서 **시내 중심 + 대표 명소 + 먹거리** 위주로 가는 게 좋아요.  
처음 가는 분 기준으로 무리 없는 일정으로 추천드릴게요.

---

## 도쿄 2박 3일 추천 코스

### 1일차: 시부야 + 하라주쿠 + 신주쿠
**핵심 분위기:** 도쿄의 젊고 활기찬 중심

- **오전/낮**: 도착 → 숙소 이동/체크인
- **오후**: 시부야
  - 시부야 스크램블 교차로
  - 시부야 스카이(예약 추천)
  - 스크램블 스퀘어 쇼핑
- **이어가기**: 하라주쿠/오모테산도
  - 다케시타 거리
  - 오모테산도 산책, 카페
- **저녁**: 신주쿠
  - 도쿄 도청사 전망대(무료)
  - 골든가이/가부키초 분위기 보기
  - 저녁식사

**포인트**
- 첫날은 이동 동선을 짧게 묶어서 피로도를 줄이는 게 좋아요
- 야경은 시부야 스카이 또는 도쿄 도청사 중 하나만 선택해도 충분합니다

---

### 2일차: 아사쿠사 + 우에노 + 아키하바라
**핵심 분위기:** 전통, 시장, 전자/서브컬처

- **오전**: 아사쿠사
  - 센소지
  - 카미나리몬
  - 나카미세 거리
- **점심**: 아사쿠사에서 텐동, 돈카츠, 소바 등
- **오후**: 우에노
  - 우에노 공원
  - 아메요코 시장
  - 박물관/미술관/동물원 중 택1
- **저녁**: 아키하바라
  - 전자상가 구경
  - 애니메이션 숍, 게임센터
  - 가볍게 저녁/간식

**포인트**
- 도쿄의 전통과 현대 분위기를 하루에 같이 볼 수 있는 코스예요
- 쇼핑보다 “도쿄 구경” 느낌이 강해서 처음 여행에 좋아요

---

### 3일차: 긴자 + 츠키지 + 도쿄역 근처 / 귀국
**핵심 분위기:** 마지막 날 가볍게, 맛집과 쇼핑

- **아침**: 츠키지 장외시장
  - 초밥, 해산물 덮밥, 계란말이
- **오전**: 긴자
  - 백화점 쇼핑
  - 카페
  - 기념품 구입
- **점심 후**: 도쿄역 주변
  - 도쿄 캐릭터 스트리트
  - 야에스 지역
- **이동**: 공항으로 출발

**포인트**
- 귀국일은 공항 이동 시간을 넉넉하게 잡으세요
- 나리타인지 하네다인지에 따라 일정이 크게 달라집니다

---

## 2박 3일 숙소 추천
- **신주쿠**: 교통 편하고 이동하기 가장 무난
- **시부야**: 분위기 좋고 젊은 감성
- **우에노**: 가성비 좋고 공항 접근성도 괜찮음

처음 도쿄 가면 **신주쿠 or 우에노**가 특히 편해요.

---

## 2박 3일 여행 팁
- 너무 많은 곳을 넣기보다 **하루 2~3개 지역**만 보는 게 좋아요
- **시부야 스카이, 인기 맛집, 디즈니 관련**은 예약 추천
- 교통은 **스이카/파스모 카드** 하나 있으면 편해요
- 도쿄는 **걷는 양이 많아서 편한 신발**이 중요합니다

---

원하시면 제가 이 코스를  
1. **커플용**  
2. **친구랑 가는 코스**  
3. **혼자 여행용**  
4. **맛집 중심**  
5. **예산 1인 기준**  

중 하나로 더 딱 맞게 다시 짜드릴게요.

--------------------------------------------------
🧑 User: : q

👋 대화를 종료합니다.


## 4. 스트리밍 응답 (Streaming)

### 4-1. AIMessageChunk

- `invoke()`는 전체 `AIMessage`를 한번에 반환합니다
- `stream()`은 토큰단위로 `AIMessageChunk`를 순차적으로 반환합니다
- 실시간 응답 표시가 가능합니다
- 사용자 경험을 개선합니다

In [13]:
for chunk in llm.stream("Python에 대해 설명해주세요."):
    # print(chunk)
    print(chunk.text, end="", flush=True)
print("\n")

Python은 **읽기 쉽고 배우기 쉬운 범용 프로그래밍 언어**입니다.  
웹 개발, 데이터 분석, 인공지능, 자동화, 과학 계산 등 아주 다양한 분야에서 널리 사용됩니다.

## Python의 특징
- **문법이 간단함**: 다른 언어보다 코드가 짧고 직관적입니다.
- **가독성이 좋음**: 사람이 읽기 쉽게 설계되었습니다.
- **범용성**: 웹, 앱, 게임, 데이터 처리, AI 등 여러 분야에서 활용됩니다.
- **풍부한 라이브러리**: 필요한 기능을 쉽게 가져다 쓸 수 있습니다.
- **활발한 커뮤니티**: 자료가 많고 문제 해결이 비교적 쉽습니다.

## Python 예시
```python
print("Hello, world!")
```

이 코드는 화면에 `Hello, world!`를 출력합니다.

## 간단한 변수 예시
```python
name = "철수"
age = 20

print(name)
print(age)
```

## Python이 많이 쓰이는 분야
- **웹 개발**: Django, Flask
- **데이터 분석**: Pandas, NumPy
- **인공지능/머신러닝**: TensorFlow, PyTorch
- **자동화**: 반복 작업을 자동으로 처리
- **교육용 언어**: 프로그래밍 입문에 적합

## 장점
- 배우기 쉬움
- 빠르게 개발 가능
- 다양한 분야에 적용 가능

## 단점
- C/C++ 같은 언어보다 실행 속도가 느릴 수 있음
- 모바일 앱이나 초고성능 시스템에는 덜 자주 쓰임

원하시면 제가 다음 중 하나로 이어서 설명해드릴 수 있습니다:
1. **Python 문법 기초**
2. **Python 설치 방법**
3. **Python으로 할 수 있는 일**
4. **Python 예제 코드 더 보기**



### 4-2. AIMessageChunk 결합

여러 개의 청크를 합치면 AIMessage와 동일한 구조가 됩니다. `+` 연산자로 `invoke()`처럼 전체 메시지를 복원할 수 있습니다.

In [14]:
full_message = None

for chunk in llm.stream("인공지능이 무엇인가요?"):
    # 청크 결합
    full_message = chunk if full_message is None else full_message + chunk
    print(chunk.text, end="", flush=True)

인공지능(AI)은 **사람처럼 배우고, 판단하고, 문제를 해결하도록 만든 컴퓨터 기술**입니다.

쉽게 말하면:
- **데이터를 보고 패턴을 찾고**
- **그 패턴을 바탕으로 예측하거나 결정하고**
- **대화, 번역, 이미지 인식, 추천** 같은 일을 할 수 있습니다.

예를 들면:
- 음성 비서가 말을 알아듣는 것
- 유튜브/넷플릭스가 영상을 추천하는 것
- 사진 속 얼굴이나 물체를 찾는 것
- 챗봇이 질문에 답하는 것

인공지능은 크게 두 가지로 볼 수 있습니다:
1. **좁은 인공지능(약한 AI)**: 특정 작업만 잘하는 AI  
   - 예: 번역기, 추천 시스템
2. **일반 인공지능(강한 AI)**: 인간처럼 다양한 일을 폭넓게 수행하는 AI  
   - 현재는 아직 완전히 구현되지 않았습니다.

원하시면 제가 **“인공지능이 작동하는 방식”**도 아주 쉽게 설명해드릴게요.

In [15]:
print(f"\n메시지 타입: {type(full_message)}")
print(f"전체 content: {full_message.content}")


메시지 타입: <class 'langchain_core.messages.ai.AIMessageChunk'>
전체 content: 인공지능(AI)은 **사람처럼 배우고, 판단하고, 문제를 해결하도록 만든 컴퓨터 기술**입니다.

쉽게 말하면:
- **데이터를 보고 패턴을 찾고**
- **그 패턴을 바탕으로 예측하거나 결정하고**
- **대화, 번역, 이미지 인식, 추천** 같은 일을 할 수 있습니다.

예를 들면:
- 음성 비서가 말을 알아듣는 것
- 유튜브/넷플릭스가 영상을 추천하는 것
- 사진 속 얼굴이나 물체를 찾는 것
- 챗봇이 질문에 답하는 것

인공지능은 크게 두 가지로 볼 수 있습니다:
1. **좁은 인공지능(약한 AI)**: 특정 작업만 잘하는 AI  
   - 예: 번역기, 추천 시스템
2. **일반 인공지능(강한 AI)**: 인간처럼 다양한 일을 폭넓게 수행하는 AI  
   - 현재는 아직 완전히 구현되지 않았습니다.

원하시면 제가 **“인공지능이 작동하는 방식”**도 아주 쉽게 설명해드릴게요.


### 📖 과제 2: 멀티턴 대화 스트리밍으로 긴 설명 받기

멀티턴 대화 방식으로 AI 응답을 스트리밍 방식(streaming)으로 출력하도록 구현하세요.

1. 멀티턴 대화 유지
- 이전 대화 내용을 conversation 리스트로 관리할 것
- 사용자 입력과 AI 응답이 모두 누적되어야 함
2. 스트리밍 응답 처리
- llm.invoke() 대신 llm.stream() 사용
- AI 응답을 토큰 단위로 실시간 출력할 것
3. 응답 저장
- 스트리밍으로 출력된 내용을 하나의 문자열로 합쳐서
- AIMessage로 conversation에 저장할 것


💡 힌트: 3-2. 멀티턴 에서 진행한 코드를 베이스로 사용하세요!

In [16]:
# CODE HERE

---

### 참고 자료

- [LangChain Models 공식 문서](https://docs.langchain.com/oss/python/langchain/models)
- [LangChain Messages 공식 문서](https://docs.langchain.com/oss/python/langchain/messages)
- [Chat Models 통합 문서](https://docs.langchain.com/oss/python/integrations/chat)